# Laboratório 11: Detecção e Tratamento Integrado de Outliers
**Disciplina:** Extração e Preparação de Dados (IBM8915)
**Professor:** Luís Aramis

**Objetivo:** Atuar como um Auditor de Risco. Você vai aprender a diagnosticar anomalias univariadas usando Boxplots e a regra do IQR, tratar esses extremos de forma segura com `clip()` (Winsorization) e, por fim, utilizar Inteligência Artificial (Isolation Forest) para encontrar fraudes ocultas no cruzamento de múltiplas variáveis.

In [ ]:
# Importação das bibliotecas essenciais
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Simulando um dataset de E-commerce com anomalias plantadas
np.random.seed(42)
n_linhas = 1000

df_risco = pd.DataFrame({
    'ID_Cliente': range(1, n_linhas + 1),
    'Idade_Conta': np.random.randint(10, 3650, n_linhas), # Dias
    'Receita': np.random.exponential(scale=500, size=n_linhas) + 50, 
    'Ticket_Medio': np.random.normal(150, 40, n_linhas)
})

# Plantando Outliers Univariados (O "Elon Musk")
df_risco.loc[990:995, 'Receita'] = 50000

# Plantando Outliers Multivariados (Fraudes Complexas)
# Contas recém-criadas (10 dias) com Ticket Médio absurdamente alto, 
# mas que não rompem o limite univariado isoladamente.
df_risco.loc[996:999, 'Idade_Conta'] = 10
df_risco.loc[996:999, 'Ticket_Medio'] = 500

display(df_risco.tail(15))

## Parte 1: O Raio-X Visual (Boxplot)
Antes de fazermos matemática, precisamos enxergar o problema.
**Tarefa:** Plote um gráfico de caixa (`sns.boxplot`) para a coluna `Receita`. Identifique visualmente a caixa (onde estão 50% dos dados) e os pontos que representam os outliers (fora dos bigodes).

In [ ]:
# ESCREVA SEU CÓDIGO AQUI
# Plote o boxplot da coluna 'Receita' usando seaborn

plt.figure(figsize=(10, 4))
sns.boxplot(x=df_risco['Receita'], color='steelblue')
plt.title('Boxplot da Receita (Outliers Univariados)')
plt.xlabel('Receita (R$)')
plt.tight_layout()
plt.show()

## Parte 2: O Cálculo das Cercas (Regra do IQR)
O gráfico nos mostrou que temos valores extremos, mas onde exatamente começa a anomalia? 
A regra do Intervalo Interquartil (IQR) dita que:
1. Calculamos o Q1 (Percentil 25%) e o Q3 (Percentil 75%).
2. Subtraímos um do outro para achar o IQR (`Q3 - Q1`).
3. O Limite Superior é `Q3 + 1.5 * IQR`. Tudo acima disso é outlier matemático!

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# 1. Calcule Q1 e Q3 da coluna 'Receita' usando o método .quantile()
Q1 = df_risco['Receita'].quantile(0.25)
Q3 = df_risco['Receita'].quantile(0.75)

# 2. Calcule o IQR
IQR = Q3 - Q1

# 3. Defina o Limite Superior
limite_superior = Q3 + 1.5 * IQR

print(f"Q1: R$ {Q1:.2f}")
print(f"Q3: R$ {Q3:.2f}")
print(f"IQR: R$ {IQR:.2f}")
print(f"O Limite Máximo Aceitável para a Receita é: R$ {limite_superior:.2f}")

## Parte 3: A Cirurgia de Clipping
Se usarmos um `dropna()` e deletarmos os milionários da base, perderemos dados valiosos de `Idade_Conta` e `Ticket_Medio` que pertencem a eles. 

**A Solução:** Vamos usar a técnica de *Winsorization*.
**Sua Tarefa:** Use o método nativo `.clip()` do Pandas na coluna `Receita`. Defina o parâmetro `upper` como o `limite_superior` que você acabou de calcular. Salve isso em uma nova coluna chamada `Receita_Tratada`.

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# Aplique o .clip() para amassar a cauda longa
df_risco['Receita_Tratada'] = df_risco['Receita'].clip(upper=limite_superior)

# Prove que funcionou verificando o novo valor máximo da coluna tratada
print("Máximo original:", df_risco['Receita'].max())
print("Novo Máximo (após clip):", df_risco['Receita_Tratada'].max())

## Parte 4: A Visão Multivariada
Nós resolvemos a Receita, mas e as fraudes ocultas? Existem clientes com contas recém-criadas gastando fortunas de uma só vez, o que burla o IQR isolado.

Vamos invocar o Scikit-Learn e a classe `IsolationForest` para auditar nossa base cruzando 3 colunas simultaneamente.

In [ ]:
# 1. Isole as features que vão alimentar o modelo de IA
features = ['Idade_Conta', 'Receita_Tratada', 'Ticket_Medio']
X = df_risco[features]

# 2. Instancie o Detetive (Isolation Forest) esperando 1% de fraudes (contamination=0.01)
detector = IsolationForest(contamination=0.01, random_state=42)

# 3. Encaixe o modelo e faça as predições (fit_predict)
df_risco['Outlier_ML'] = detector.fit_predict(X)

# Faça um value_counts da nova coluna 'Outlier_ML' para ver quantos '-1' foram encontrados!
print("Distribuição de Normal (1) vs Anomalia (-1):")
print(df_risco['Outlier_ML'].value_counts())

## Parte 5: O Desafio do Auditor (Para Casa)
O modelo Isolation Forest avaliou múltiplas dimensões e marcou os dados normais como `1` e as anomalias (fraudes) como `-1`.

Como analista de risco, seu diretor pediu um relatório comprovando o porquê o modelo barrou essas contas.

**Sua Tarefa (Missão de Casa):**
1. **Isolar:** Crie um filtro no `df_risco` para visualizar apenas os clientes classificados como `-1`.
2. **Comparar Perfis:** Use `df_risco.groupby('Outlier_ML').mean()` para comparar as médias de `Idade_Conta`, `Receita_Tratada` e `Ticket_Medio` dos clientes normais (1) vs anômalos (-1).
3. **Visão Executiva (Gráfico):** Crie um gráfico de dispersão (`sns.scatterplot`) colocando `Idade_Conta` no eixo X e `Ticket_Medio` no eixo Y. Use o parâmetro `hue='Outlier_ML'` para colorir os pontos e evidenciar visualmente as fraudes detectadas isoladas do comportamento padrão.
4. **Conclusão Auditorial:** Escreva qual é o padrão suspeito (o modus operandi) dessa fraude complexa.

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# 1. Isolar anomalias
anomalias = df_risco[df_risco['Outlier_ML'] == -1]
print(f"Total de anomalias detectadas: {len(anomalias)}")
display(anomalias[['ID_Cliente', 'Idade_Conta', 'Receita_Tratada', 'Ticket_Medio', 'Outlier_ML']])

# 2. Comparar Médias (groupby)
print("\nPerfil Médio Normal (1) vs Anômalo (-1):")
display(df_risco.groupby('Outlier_ML')[['Idade_Conta', 'Receita_Tratada', 'Ticket_Medio']].mean())

# 3. Gráfico (Scatterplot de cruzamento Idade x Ticket com hue=Outlier_ML)
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df_risco,
    x='Idade_Conta',
    y='Ticket_Medio',
    hue='Outlier_ML',
    palette={1: 'steelblue', -1: 'red'},
    alpha=0.7
)
plt.title('Detecção de Fraudes: Idade da Conta vs Ticket Médio')
plt.xlabel('Idade da Conta (dias)')
plt.ylabel('Ticket Médio (R$)')
plt.legend(title='Outlier_ML', labels=['Normal (1)', 'Fraude (-1)'])
plt.tight_layout()
plt.show()

# 4. Padrão das contas barradas:
print("""
Padrão Suspeito (Modus Operandi):
As contas anomalas possuem Idade_Conta muito baixa (contas recém-criadas, ~10 dias)
combinada com Ticket_Médio muito alto (>R$500). Isso indica contas criadas recentemente
com comportamento de compras atipicamente alto, sugerindo fraude ou lavagem de dinheiro.
""")

## Parte 6: Desafio com Dados Reais (Kaggle)
Até agora usamos dados simulados. Como seria aplicar isso no mundo real?

Vamos usar um dataset famoso do Kaggle: **Credit Card Fraud Detection**.
Ele contém transações de cartões de crédito, onde a classe positiva (fraude) é altamente desbalanceada.
Link do Dataset no Kaggle: [Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud)

Baixaremos uma amostra ou usaremos um link raw disponível no GitHub.

In [ ]:
# Carregando o dataset real de fraudes em cartão de crédito (Kaggle)
# Vamos usar um link direto (RAW), pode demorar alguns segundos devido ao tamanho do arquivo original.
url_kaggle = 'https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv'

# Lendo apenas as primeiras 50 mil linhas para não sobrecarregar a memória nesta aula
df_kaggle = pd.read_csv(url_kaggle, nrows=50000)

display(df_kaggle.head())
print("Formato dos dados: ", df_kaggle.shape)

### Exercício 1: Explorando a Distribuição Visual
As variáveis `V1` a `V28` são o resultado de uma transformação PCA (para manter a privacidade). As únicas variáveis originais são `Time` e `Amount` (Valor da Transação).

**Sua Tarefa:**
1. Crie um Boxplot usando Seaborn para a variável `Amount`.
2. Analise: Pelo aspecto visual da cauda (pontos fora da caixa), o método do IQR (que corta os excessos) parece uma boa ideia para descobrir anomalias nesta coluna isolada?

In [ ]:
# ESCREVA SEU CÓDIGO AQUI
# Plote o Boxplot da coluna 'Amount'
plt.figure(figsize=(10, 4))
sns.boxplot(x=df_kaggle['Amount'], color='steelblue')
plt.title('Boxplot do Valor das Transações (Amount)')
plt.xlabel('Amount (€)')
plt.tight_layout()
plt.show()

### Exercício 2: O Custo do Falso Positivo na Regra do IQR
Muitas fraudes podem ocorrer em transações de alto valor. Se usarmos a matemática do IQR cegamente, podemos estar punindo (falso positivo) clientes legítimos gastando muito. 
No dataset do Kaggle, a coluna `Class` vale 1 para fraude e 0 para transação normal.

**Sua Tarefa:**
1. Calcule o `limite_superior` para a coluna `Amount` usando a Regra do IQR (Q3 + 1.5 * IQR).
2. Descubra: Das transações que ficaram **acima** desse limite superior, quantas eram transações totalmente **Legítimas** (`Class == 0`)?

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

# 1. Calcule o Limite Superior
Q1 = df_kaggle['Amount'].quantile(0.25)
Q3 = df_kaggle['Amount'].quantile(0.75)
IQR = Q3 - Q1
lim_sup = Q3 + 1.5 * IQR
print(f"Limite Superior IQR para Amount: {lim_sup:.2f}")

# 2. Filtre e conte as legítimas (Class == 0) que passaram do lim_sup
acima_limite = df_kaggle[df_kaggle['Amount'] > lim_sup]
legitimas_barradas = acima_limite[acima_limite['Class'] == 0]
print(f"\nTransações acima do limite: {len(acima_limite)}")
print(f"Das quais são LEGÍTIMAS (Class==0): {len(legitimas_barradas)}")
print(f"Isso representa {len(legitimas_barradas)/len(acima_limite)*100:.1f}% de falsos positivos!")

### Exercício 3: O Detetive de IA Multivariado (Isolation Forest)
Já que uma variável só não resolveu, vamos invocar o modelo de Machine Learning ensinando-o a ler 29 variáveis simultaneamente.

**Sua Tarefa:**
1. Isole a base de aprendizado ignorando as colunas `Class` (pois o detector é não-supervisionado) e `Time`: 
`X = df_kaggle.drop(columns=['Class', 'Time'])`
2. Instancie e treine o `IsolationForest` com `contamination=0.01` e `random_state=42`.
3. Salve o `.fit_predict(X)` em uma nova coluna `Outlier_ML` no dataset original.
4. Veja na prática a eficácia: Faça um cruzamento (`pd.crosstab`) da nossa predição `Outlier_ML` com a realidade `Class` do Kaggle!

In [ ]:
# ESCREVA SEU CÓDIGO AQUI

X = df_kaggle.drop(columns=['Class', 'Time'])
detector_kaggle = IsolationForest(contamination=0.01, random_state=42)
df_kaggle['Outlier_ML'] = detector_kaggle.fit_predict(X)

# Mostre a confusão:
print("Crosstab: Predição do Isolation Forest vs Realidade (Class):")
display(pd.crosstab(df_kaggle['Outlier_ML'], df_kaggle['Class']))